In [ ]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
from IPython.display import VimeoVideo
from pymongo import MongoClient
from sklearn.metrics import mean_absolute_error
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.ar_model import AutoReg

warnings.simplefilter(action="ignore", category=FutureWarning)


In [ ]:
VimeoVideo("665851858", h="e39fc3d260", width=600)


In [ ]:
host = "192.98.213.2"


In [ ]:
VimeoVideo("665851852", h="16aa0a56e6", width=600)


In [ ]:
client = MongoClient(host=host, port=27017)
db = client["air-quality"]
nairobi = db["nairobi"]


In [ ]:
VimeoVideo("665851840", h="e048425f49", width=600)


In [ ]:
def wrangle(collection):
    results = collection.find(
        {"metadata.site": 29, "metadata.measurement": "P2"},
        projection={"P2": 1, "timestamp": 1, "_id": 0},
    )

    # Read data into DataFrame
    df = pd.DataFrame(list(results)).set_index("timestamp")

    # Localize timezone
    df.index = df.index.tz_localize("UTC").tz_convert("Africa/Nairobi")

    # Remove outliers
    df = df[df["P2"] < 500]

    # Resample to 1hr window
    y = df["P2"].resample("1H").mean().fillna(method='ffill')

    return y


In [ ]:
y = wrangle(nairobi)
y.head()


In [ ]:
# Check your work
assert isinstance(y, pd.Series), f"`y` should be a Series, not type {type(y)}"
assert len(y) == 2928, f"`y` should have 2928 observations, not {len(y)}"
assert y.isnull().sum() == 0


In [ ]:
VimeoVideo("665851830", h="85f58bc92b", width=600)


In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
plot_acf(y, ax=ax)
plt.xlabel("Lag [hours]")
plt.ylabel("Correlation Coefficient");


In [ ]:
VimeoVideo("665851811", h="ee3a2b5c24", width=600)


In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
plot_pacf(y,ax=ax)
plt.xlabel("Lag [hours]")
plt.ylabel("Correlation Coefficient");


In [ ]:
VimeoVideo("665851798", h="6c191cd94c", width=600)


In [ ]:
cutoff_test = int(len(y) * 0.95)

y_train = y.iloc[: cutoff_test]
y_test = y.iloc[cutoff_test :]
#len(y_test) + len(y_train) == len(y)


In [ ]:
y_train_mean = y_train.mean()
y_pred_baseline = [y_train_mean] * len(y_train)
mae_baseline = mean_absolute_error(y_train, y_pred_baseline)

print("Mean P2 Reading:", round(y_train_mean, 2))
print("Baseline MAE:", round(mae_baseline, 2))


In [ ]:
VimeoVideo("665851769", h="94a4296cde", width=600)


In [ ]:
model = AutoReg(y_train, lags=26).fit()


In [ ]:
VimeoVideo("665851746", h="1a4511e883", width=600)


In [ ]:
y_pred = model.predict().dropna()
training_mae = mean_absolute_error(y_train.iloc[26:],y_pred)
print("Training MAE:", training_mae)


In [ ]:
VimeoVideo("665851744", h="60d053b455", width=600)


In [ ]:
y_train_resid = model.resid
y_train_resid.tail()


In [ ]:
VimeoVideo("665851712", h="9ff0cdba9c", width=600)


In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
y_train_resid.plot(ylabel="Residual value",ax=ax)


In [ ]:
VimeoVideo("665851702", h="b494adc297", width=600)


In [ ]:
y_train_resid.hist()
plt.xlabel("Residual value")
plt.ylabel("Frequency")
plt.title("AR(26), Distribution of residual")


In [ ]:
VimeoVideo("665851684", h="d6d782a1f3", width=600)


In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
plot_acf(y_train_resid,ax=ax);


In [ ]:
VimeoVideo("665851662", h="72e767e121", width=600)


In [ ]:
y_pred_test = model.predict(y_test.index.min(), y_test.index.max())
test_mae = mean_absolute_error(y_test,y_pred_test) # Model.predict(start, end)
print("Test MAE:", test_mae)


In [ ]:
df_pred_test = pd.DataFrame(
    {"y_test": y_test, "y_pred": y_pred_test}, index=y_test.index
)


In [ ]:
VimeoVideo("665851628", h="29b43e482e", width=600)


In [ ]:
fig = px.line(df_pred_test, labels={"value": "P2"})
fig.show()


In [ ]:
VimeoVideo("665851599", h="bb30d96e43", width=600)


In [ ]:
%%capture

y_pred_wfv = pd.Series()
history = y_train.copy()
for i in range(len(y_test)):
    model = AutoReg(history,lags=26).fit()
    next_pred = model.forecast()
    y_pred_wfv = y_pred_wfv.append(next_pred)
    history = history.append(y_test[next_pred.index])


In [ ]:
VimeoVideo("665851568", h="a764ab5416", width=600)


In [ ]:
test_mae = mean_absolute_error(y_test,y_pred_wfv)
print("Test MAE (walk forward validation):", round(test_mae, 2))


In [ ]:
VimeoVideo("665851553", h="46338036cc", width=600)


In [ ]:
print(model.params)


In [ ]:
VimeoVideo("665851529", h="39284d37ac", width=600)


In [ ]:
df_pred_test = pd.DataFrame(
    {"y_test": y_test,"y_pred_wfv": y_pred_wfv}
    )

fig = px.line(df_pred_test, labels={"value": "PM2.5"})
fig.show()
